In [ ]:
# ==============================
# 1. INSTALL DEPENDENCIES
# ==============================
!pip install -q lightgbm catboost imbalanced-learn xgboost scikit-learn pandas numpy openpyxl

In [ ]:
# ==============================
# 2. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef, cohen_kappa_score, confusion_matrix)

from imblearn.metrics import geometric_mean_score
from imblearn.over_sampling import ADASYN
from catboost import CatBoostClassifier

from sklearn.linear_model import LogisticRegression
!pip install lime

In [ ]:
# ==============================
# 3. LOAD DATASET
# ==============================
df = pd.read_csv('/content/INCART 2-lead Arrhythmia Database.csv')


In [ ]:
# ==============================
# 4. PREPROCESSING
# ==============================

# Remove column if exists
df = df.drop('duration', axis=1, errors='ignore')

# Handle unknown values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

# Convert categorical columns
df = pd.get_dummies(df, drop_first=True)

# Select target column (last column)
target_column = df.columns[-1]

# Features and Target
X = df.drop(target_column, axis=1)
y = df[target_column]


In [ ]:
# ==============================
# 5. SPLIT DATA
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ==============================
# 6. FEATURE SCALING
# ==============================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ADASYN
adasyn = ADASYN(random_state=42)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train_scaled, y_train)

In [ ]:
# ==============================
# 6. METRIC FUNCTIONS
# ==============================
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"Accuracy    : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision   : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall      : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Specificity : {tn / (tn + fp):.4f}")
    print(f"F1 Score    : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"ROC-AUC     : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"MCC         : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"G-Mean      : {geometric_mean_score(y_true, y_pred):.4f}")
    print(f"Kappa       : {cohen_kappa_score(y_true, y_pred):.4f}")

# **CatBoostClassifier**

In [ ]:
# ==============================
# 7. HYPERPARAMETER TUNING
# ==============================
params = {
    "iterations":[100,200],
    "depth":[4,5],
    "learning_rate":[0.03],
    "class_weights":[[1,2],[1,3]]
}

cv_small = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rscv = RandomizedSearchCV(
    CatBoostClassifier(verbose=0),
    params,
    n_iter=2,
    scoring='f1',
    cv=cv_small,
    n_jobs=-1
)

rscv.fit(X_train_ad, y_train_ad)
best = rscv.best_estimator_

print("🔥 Tuned Model Performance (Test Set)")
y_pred = best.predict(X_test_scaled)
y_prob = best.predict_proba(X_test_scaled)[:,1]
compute_metrics(y_test, y_pred, y_prob)

print("Best Params:", rscv.best_params_)


In [ ]:
# ==============================
# 8. CROSS VALIDATION (MAIN)
# ==============================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
results1 = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best.predict(X_test_f)
    y_prob = best.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results1.append({
        "Fold": fold+1,
        "Classifier": "CatBoost",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [ ]:
# ==============================
# 9. PRINT RESULTS
# ==============================
df_results = pd.DataFrame(results1)

print("\n===== Fold Results =====")
print(df_results.to_string(index=False))

print("\n===== Average Results =====")
print(df_results.mean(numeric_only=True))


# ==============================
# 10. SAVE TO EXCEL
# ==============================
df_results.to_excel("catboost_results.xlsx", index=False)

print("\n✅ Results saved to catboost_results.xlsx")

In [ ]:
# ==============================
# SHAP EXPLAINABILITY (FINAL FIX)
# ==============================

print("\n--- SHAP EXPLANATION ---")

import shap
import matplotlib.pyplot as plt

# Explainer
explainer = shap.TreeExplainer(best)

# SHAP values
shap_values = explainer.shap_values(X_test)

# ✅ Handle classification
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

# -------------------------------
# 1. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_test, plot_type="bar")

# -------------------------------
# 2. DETAILED SUMMARY PLOT
# -------------------------------
shap.summary_plot(shap_values_class1, X_test)

# -------------------------------
# 3. DEPENDENCE PLOT
# -------------------------------
shap.dependence_plot(
    X_test.columns[0],
    shap_values_class1,
    X_test
)

# -------------------------------
# 4. LOCAL EXPLANATION (FORCE PLOT)
# -------------------------------
base_value = explainer.expected_value
if isinstance(base_value, list):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_test.iloc[0],
    matplotlib=True
)

# **LIME EXPLANATION (CatBoostClassifier)**

In [ ]:
# ==========================================
# INSTALL LIME
# ==========================================
!pip install -q lime

# ==========================================
# IMPORT LIBRARIES
# ==========================================
import numpy as np
import pandas as pd

from lime.lime_tabular import LimeTabularExplainer
from IPython.display import HTML, display

print("\n--- LIME EXPLANATION (CatBoost) ---")

# ==========================================
# CONVERT DATA TO NUMPY
# ==========================================
X_train_np = X_train.values
X_test_np  = X_test.values

# ==========================================
# CREATE LIME EXPLAINER
# ==========================================
lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X_train.columns.tolist(),
    class_names=['No', 'Yes'],
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

# ==========================================
# EXPLAIN FIRST TEST SAMPLE
# ==========================================
lime_exp = lime_explainer.explain_instance(
    data_row=X_test_np[0],
    predict_fn=best.predict_proba,   # CatBoost model
    num_features=10
)

# ==========================================
# DISPLAY EXPLANATION
# ==========================================
display(HTML(lime_exp.as_html()))

# ==========================================
# SAVE HTML FILE
# ==========================================
lime_exp.save_to_file("lime_catboost.html")

print("\n✅ LIME explanation saved as 'lime_catboost.html'")

# **LightGBM**

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
# ==============================
# LIGHTGBM TUNING
# ==============================
params_lgb = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [31, 50],
    "max_depth": [-1, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

rscv_lgb = RandomizedSearchCV(
    LGBMClassifier(class_weight='balanced'),
    params_lgb,
    n_iter=2,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_lgb.fit(X_train_ad, y_train_ad)

best_lgb = rscv_lgb.best_estimator_

In [ ]:
print("\n LightGBM (Tuned)")

y_pred_lgb = best_lgb.predict(X_test_scaled)
y_prob_lgb = best_lgb.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_lgb, y_prob_lgb)

print("Best Params (LGB):", rscv_lgb.best_params_)

In [ ]:
results_lgb = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling (optional but keeping consistent)
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_lgb.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_lgb.predict(X_test_f)
    y_prob = best_lgb.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_lgb.append({
        "Fold": fold+1,
        "Classifier": "LightGBM",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [ ]:
df_lgb = pd.DataFrame(results_lgb)

final_df = pd.concat([ df_lgb])

In [ ]:
print("\n===== LIGHTGBM  MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLAINABILITY (LightGBM FINAL)
# ==============================

print("\n--- SHAP EXPLANATION (LightGBM) ---")

import shap
import numpy as np

# Explainer
explainer = shap.TreeExplainer(best_lgb)

# SHAP values
shap_values = explainer.shap_values(X_test)

# ✅ Handle classification (binary)
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

# ✅ Ensure 2D shape
if len(shap_values_class1.shape) == 1:
    shap_values_class1 = shap_values_class1.reshape(1, -1)

# -------------------------------
# 1. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_test, plot_type="bar")

# -------------------------------
# 2. DETAILED SUMMARY
# -------------------------------
shap.summary_plot(shap_values_class1, X_test)

# -------------------------------
# 3. DEPENDENCE PLOT
# -------------------------------
shap.dependence_plot(
    X_test.columns[0],
    shap_values_class1,
    X_test
)

# -------------------------------
# 4. LOCAL EXPLANATION (FORCE PLOT)
# -------------------------------
base_value = explainer.expected_value
if isinstance(base_value, list):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_test.iloc[0],
    matplotlib=True
)

# **LIME EXPLANATION (LightGBM)**

In [ ]:
# ==============================
# LIME EXPLANATION (LightGBM)
# ==============================

# 🔥 Run once if needed:
# !pip install lime

print("\n--- LIME EXPLANATION (LightGBM) ---")

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

# -------------------------------
# 1. Prepare data (same as training)
# -------------------------------
X_train_np = np.array(X_train)   # use same format used in model
X_test_np  = np.array(X_test)

# -------------------------------
# 2. Create LIME Explainer
# -------------------------------
lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X_train.columns,     # correct feature names
    class_names=['No', 'Yes'],         # ⚠️ adjust based on your model
    mode='classification'
)

# -------------------------------
# 3. Explain one instance
# -------------------------------
lime_exp = lime_explainer.explain_instance(
    X_test_np[0],                     # first test sample
    best_lgb.predict_proba,           # LightGBM model
    num_features=10
)

# -------------------------------
# 4. Show explanation
# -------------------------------
lime_exp.show_in_notebook(show_table=True)

# -------------------------------
# 5. (Optional) Save as HTML
# -------------------------------
lime_exp.save_to_file("lime_lightgbm.html")

# **XGBoost**

In [ ]:
from xgboost import XGBClassifier

In [ ]:
#Hyperparameter Tuning

# ==============================
# XGBOOST TUNING
# ==============================
params_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

rscv_xgb = RandomizedSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    params_xgb,
    n_iter=2,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_xgb.fit(X_train_ad, y_train_ad)

best_xgb = rscv_xgb.best_estimator_

In [ ]:
print("\n⚡ XGBoost (Tuned)")

y_pred_xgb = best_xgb.predict(X_test_scaled)
y_prob_xgb = best_xgb.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_xgb, y_prob_xgb)

print("Best Params (XGB):", rscv_xgb.best_params_)

In [ ]:
results_xgb = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_xgb.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_xgb.predict(X_test_f)
    y_prob = best_xgb.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_xgb.append({
        "Fold": fold+1,
        "Classifier": "XGBoost",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

In [ ]:
df_xgb = pd.DataFrame(results_xgb)

final_df = pd.concat([ df_xgb])

In [ ]:
print("\n===== ALL MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLANATION (XGBOOST FIXED)
# ==============================

print("\n--- SHAP EXPLANATION (XGBoost) ---")

import shap
import numpy as np

# Explainer
explainer = shap.TreeExplainer(best_xgb)

# SHAP values
shap_values = explainer.shap_values(X_test)

# ✅ Handle classification
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

# ✅ Ensure correct shape
if len(shap_values_class1.shape) == 1:
    shap_values_class1 = shap_values_class1.reshape(1, -1)

# -------------------------------
# 1. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_test, plot_type="bar")

# -------------------------------
# 2. DETAILED SUMMARY
# -------------------------------
shap.summary_plot(shap_values_class1, X_test)

# -------------------------------
# 3. DEPENDENCE PLOT
# -------------------------------
shap.dependence_plot(
    X_test.columns[0],
    shap_values_class1,
    X_test
)

# -------------------------------
# 4. LOCAL EXPLANATION (FORCE PLOT)
# -------------------------------
base_value = explainer.expected_value
if isinstance(base_value, list):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_test.iloc[0],
    matplotlib=True
)

# **LIME EXPLAINABILITY for XGBoost  **

In [ ]:
# ==============================
# FINAL LIME EXPLANATION
# ==============================

# 🔥 Install LIME (run once)
!pip install lime

print("\n--- LIME EXPLANATION ---")

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

# -------------------------------
# 1. Ensure correct data format
# -------------------------------
# Convert to numpy if needed
X_train_np = np.array(X_train_scaled)
X_test_np  = np.array(X_test_scaled)

# -------------------------------
# 2. Create LIME Explainer
# -------------------------------
lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X_train.columns,     # ✅ correct feature names
    class_names=['No', 'Yes'],         # ⚠️ order must match model output
    mode='classification'
)

# -------------------------------
# 3. Explain one instance
# -------------------------------
lime_exp = lime_explainer.explain_instance(
    X_test_np[0],                     # first sample
    best_xgb.predict_proba,
    num_features=10                  # show top features
)

# -------------------------------
# 4. Show explanation
# -------------------------------
lime_exp.show_in_notebook(show_table=True)

# -------------------------------
# 5. (Optional) Save as HTML
# -------------------------------
lime_exp.save_to_file("lime_explanation.html")

# Gradient Boosting (GBM)

In [ ]:
#Import GBM

from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
params_gbm = {
    "n_estimators": [50],     # smaller = faster
    "learning_rate": [0.1],
    "max_depth": [2]          # shallow trees = faster
}

# ------------------------------------------
# FAST CROSS VALIDATION
# ------------------------------------------
cv_fast = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

# ------------------------------------------
# FAST MODEL
# ------------------------------------------
gbm_fast = GradientBoostingClassifier(
    subsample=0.8,            # faster training
    random_state=42
)

# ------------------------------------------
# RANDOM SEARCH
# ------------------------------------------
rscv_gbm = RandomizedSearchCV(
    estimator=gbm_fast,
    param_distributions=params_gbm,
    n_iter=1,
    scoring='f1',
    cv=cv_fast,
    n_jobs=-1,
    verbose=0,
    random_state=42
)

# ------------------------------------------
# TRAIN
# ------------------------------------------
rscv_gbm.fit(X_train_ad, y_train_ad)

# ------------------------------------------
# BEST MODEL
# ------------------------------------------
best_gbm = rscv_gbm.best_estimator_

print("\n✅ Best Parameters:")
print(rscv_gbm.best_params_)

print("\n✅ Best F1 Score:")
print(round(rscv_gbm.best_score_, 4))

In [ ]:
#Test Performance

print("\n Gradient Boosting (Tuned)")

y_pred_gbm = best_gbm.predict(X_test_scaled)
y_prob_gbm = best_gbm.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_gbm, y_prob_gbm)

print("Best Params (GBM):", rscv_gbm.best_params_)

In [ ]:
results_gbm = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    # ==============================
    # SPLIT DATA
    # ==============================
    X_train_f = X.iloc[train_idx]

    X_test_f = X.iloc[test_idx]

    y_train_f = y.iloc[train_idx]

    y_test_f = y.iloc[test_idx]

    # ==============================
    # FAST OVERSAMPLING
    # ==============================
    ros = RandomOverSampler(random_state=42)

    X_train_f, y_train_f = ros.fit_resample(
        X_train_f,
        y_train_f
    )

    # ==============================
    # TRAIN
    # ==============================
    start_time = time.time()

    best_gbm.fit(X_train_f, y_train_f)

    training_time = time.time() - start_time

    # ==============================
    # PREDICT
    # ==============================
    y_pred = best_gbm.predict(X_test_f)

    # ==============================
    # FAST METRICS
    # ==============================
    acc = accuracy_score(y_test_f, y_pred)

    prec = precision_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_f,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)

    balanced_acc = (rec + specificity) / 2

    # ==============================
    # STORE RESULTS
    # ==============================
    results_gbm.append({

        "Fold": fold + 1,

        "Classifier": "HistGradientBoosting",

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "Specificity": specificity,

        "F1": f1,

        "Balanced Accuracy": balanced_acc,

        "Training Time (s)": training_time
    })

In [ ]:
df_gbm = pd.DataFrame(results_gbm)

final_df = pd.concat([df_gbm])

In [ ]:
#Print Comparison

print("\n===== GBM MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLANATION (GBM FINAL)
# ==============================

print("\n--- SHAP EXPLANATION (GBM) ---")

import shap
import numpy as np

# Explainer
explainer = shap.TreeExplainer(best_gbm)

# SHAP values
shap_values = explainer.shap_values(X_test)

# ✅ Handle classification (binary)
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
else:
    shap_values_class1 = shap_values

# ✅ Ensure 2D
if len(shap_values_class1.shape) == 1:
    shap_values_class1 = shap_values_class1.reshape(1, -1)

# -------------------------------
# 1. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_test, plot_type="bar")

# -------------------------------
# 2. DETAILED SUMMARY
# -------------------------------
shap.summary_plot(shap_values_class1, X_test)

# -------------------------------
# 3. DEPENDENCE PLOT
# -------------------------------
shap.dependence_plot(
    X_test.columns[0],
    shap_values_class1,
    X_test
)

# -------------------------------
# 4. LOCAL EXPLANATION (FORCE PLOT)
# -------------------------------
base_value = explainer.expected_value
if isinstance(base_value, list):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_test.iloc[0],
    matplotlib=True
)

# **LIME EXPLANATION (GBM)**

In [ ]:
# ==============================
# LIME EXPLANATION (GBM)
# ==============================

# 🔥 Run once if not installed
# !pip install lime

print("\n--- LIME EXPLANATION (GBM) ---")

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

# -------------------------------
# 1. Use SAME data format as training
# -------------------------------
X_train_np = np.array(X_train)   # or X_train_scaled if you used scaling
X_test_np  = np.array(X_test)

# -------------------------------
# 2. Create LIME Explainer
# -------------------------------
lime_explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X_train.columns,     # correct feature names
    class_names=['No', 'Yes'],         # ⚠️ adjust based on your model
    mode='classification'
)

# -------------------------------
# 3. Explain one instance
# -------------------------------
lime_exp = lime_explainer.explain_instance(
    X_test_np[0],                     # first test sample
    best_gbm.predict_proba,           # GBM model
    num_features=10
)

# -------------------------------
# 4. Show explanation
# -------------------------------
lime_exp.show_in_notebook(show_table=True)

# -------------------------------
# 5. Save output (optional)
# -------------------------------
lime_exp.save_to_file("lime_gbm.html")

# **Logistic Regression **


In [ ]:

# ==============================
# LOGISTIC REGRESSION TUNING
# ==============================
params_lr = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l1", "l2"]
}

rscv_lr = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, solver='liblinear'),
    params_lr,
    n_iter=2,
    scoring='f1',
    cv=3,
    n_jobs=-1
)

rscv_lr.fit(X_train_ad, y_train_ad)

best_lr = rscv_lr.best_estimator_

In [ ]:
print("\n🔥 Logistic Regression (Tuned)")

y_pred_lr = best_lr.predict(X_test_scaled)
y_prob_lr = best_lr.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_lr, y_prob_lr)

print("Best Params (LR):", rscv_lr.best_params_)

In [ ]:
results_lr = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    X_train_f, X_test_f = X.iloc[train_idx], X.iloc[test_idx]
    y_train_f, y_test_f = y.iloc[train_idx], y.iloc[test_idx]

    # Scaling
    scaler = StandardScaler()
    X_train_f = scaler.fit_transform(X_train_f)
    X_test_f  = scaler.transform(X_test_f)

    # ADASYN
    adasyn = ADASYN(random_state=42)
    X_train_f, y_train_f = adasyn.fit_resample(X_train_f, y_train_f)

    # Train
    start_time = time.time()
    best_lr.fit(X_train_f, y_train_f)
    training_time = time.time() - start_time

    # Predict
    y_pred = best_lr.predict(X_test_f)
    y_prob = best_lr.predict_proba(X_test_f)[:,1]

    # Metrics
    acc = accuracy_score(y_test_f, y_pred)
    prec = precision_score(y_test_f, y_pred, zero_division=0)
    rec = recall_score(y_test_f, y_pred, zero_division=0)
    f1 = f1_score(y_test_f, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test_f, y_pred).ravel()

    specificity = tn/(tn+fp)
    fpr = fp/(fp+tn)
    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)
    mcc = matthews_corrcoef(y_test_f, y_pred)
    kappa = cohen_kappa_score(y_test_f, y_pred)
    balanced_acc = (rec + specificity)/2

    results_lr.append({
        "Fold": fold+1,
        "Classifier": "Logistic Regression",
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })


In [ ]:
#df_cat = pd.DataFrame(results1)
df_lr  = pd.DataFrame(results_lr)

final_df = pd.concat([ df_lr])

In [ ]:

print("\n===== Logistic Regression (Fold-wise) =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLANATION (FINAL FIX)
# ==============================

import shap
import pandas as pd
import numpy as np

print("\n--- SHAP EXPLANATION (LOGISTIC REGRESSION) ---")

# -------------------------------
# SAFE DATA PREPARATION
# -------------------------------
X_test_df = pd.DataFrame(X_test, columns=X.columns)

# Ensure float type (IMPORTANT FIX)
X_test_df = X_test_df.astype(float)

# -------------------------------
# ✅ NEW SHAP EXPLAINER (FIX)
# -------------------------------
explainer = shap.Explainer(best_lr, X_test_df)

# This returns SHAP object (not raw array)
shap_values = explainer(X_test_df)

# -------------------------------
# GLOBAL IMPORTANCE
# -------------------------------
shap.plots.bar(shap_values)

# -------------------------------
# DETAILED SUMMARY
# -------------------------------
shap.plots.beeswarm(shap_values)

# -------------------------------
# -------------------------------
# FORCE PLOT (CORRECT)
# -------------------------------

shap.force_plot(
    shap_values.base_values[0],
    shap_values.values[0],
    X_test_df.iloc[0],
    matplotlib=True
)

In [ ]:
# ==============================
# LIME EXPLANATION (FINAL FIX)
# ==============================

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

print("\n--- LIME EXPLANATION ---")

# -------------------------------
# ✅ ENSURE PROPER FORMAT
# -------------------------------
# Convert to DataFrame (important for feature names)
X_train_df = pd.DataFrame(X_train, columns=X.columns)
X_test_df  = pd.DataFrame(X_test,  columns=X.columns)

# Ensure numeric + no NaN
X_train_df = X_train_df.apply(pd.to_numeric, errors='coerce').fillna(0)
X_test_df  = X_test_df.apply(pd.to_numeric, errors='coerce').fillna(0)

# Convert to numpy (LIME needs numpy)
X_train_np = X_train_df.values.astype(float)
X_test_np  = X_test_df.values.astype(float)

# -------------------------------
# ✅ CREATE EXPLAINER
# -------------------------------
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X.columns.tolist(),
    class_names=["Class 0", "Class 1"],   # adjust if needed
    mode="classification"
)

# -------------------------------
# ✅ EXPLAIN ONE INSTANCE
# -------------------------------
i = 0  # choose any test index

exp = explainer.explain_instance(
    X_test_np[i],
    best_lr.predict_proba,   # IMPORTANT: use predict_proba
    num_features=10
)

# -------------------------------
# ✅ DISPLAY OUTPUT
# -------------------------------
print("\n🔹 Prediction Probabilities:")
print(best_lr.predict_proba(X_test_np[i].reshape(1, -1)))

print("\n🔹 LIME Explanation:")
for feature, weight in exp.as_list():
    print(f"{feature}: {weight:.4f}")

# -------------------------------
# ✅ VISUAL OUTPUT (NOTEBOOK)
# -------------------------------
try:
    exp.show_in_notebook(show_table=True)
except:
    print("Notebook visualization not supported here.")

# -------------------------------
# ✅ SAVE HTML FILE
# -------------------------------
exp.save_to_file("/content/lime_explanation.html")

print("\n✅ LIME explanation saved as lime_explanation.html")

# Random Forest

In [ ]:
#Import Random Forest
from sklearn.ensemble import RandomForestClassifier

In [ ]:
#Hyperparameter Tuning
# ==============================
# RANDOM FOREST TUNING
# ==============================
params_rf = {
    "n_estimators": [80, 120],
    "max_depth": [5, 10],
    "min_samples_split": [2],
    "min_samples_leaf": [1],
    "max_features": ["sqrt"]
}

# Faster CV
cv_fast = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

# Faster Random Forest
rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# Randomized Search
rscv_rf = RandomizedSearchCV(
    estimator=rf,
    param_distributions=params_rf,
    n_iter=1,              # very fast
    scoring='f1',
    cv=cv_fast,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

rscv_rf.fit(X_train_ad, y_train_ad)

best_rf = rscv_rf.best_estimator_


In [ ]:
#Test Performance
print("\n🌲 Random Forest (Tuned)")

y_pred_rf = best_rf.predict(X_test_scaled)
y_prob_rf = best_rf.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_rf, y_prob_rf)

print("Best Params (RF):", rscv_rf.best_params_)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
results_rf = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    # Split Data
    X_train_f = X.iloc[train_idx]
    X_test_f  = X.iloc[test_idx]

    y_train_f = y.iloc[train_idx]
    y_test_f  = y.iloc[test_idx]

    # ==============================
    # RANDOM OVER SAMPLING (FASTER)
    # ==============================
    ros = RandomOverSampler(random_state=42)

    X_train_f, y_train_f = ros.fit_resample(
        X_train_f,
        y_train_f
    )

    # ==============================
    # TRAIN
    # ==============================
    start_time = time.time()

    best_rf.fit(X_train_f, y_train_f)

    training_time = time.time() - start_time

    # ==============================
    # PREDICT
    # ==============================
    y_pred = best_rf.predict(X_test_f)

    y_prob = best_rf.predict_proba(X_test_f)[:, 1]

    # ==============================
    # METRICS
    # ==============================
    acc = accuracy_score(y_test_f, y_pred)

    prec = precision_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_f,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)

    fpr = fp / (fp + tn)

    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)

    mcc = matthews_corrcoef(y_test_f, y_pred)

    kappa = cohen_kappa_score(y_test_f, y_pred)

    balanced_acc = (rec + specificity) / 2

    # ==============================
    # STORE RESULTS
    # ==============================
    results_rf.append({

        "Fold": fold + 1,

        "Classifier": "Random Forest",

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "Specificity": specificity,

        "F1": f1,

        "GM": gm,

        "FPR": fpr,

        "AUC": auc,

        "MCC": mcc,

        "Kappa": kappa,

        "Balanced Accuracy": balanced_acc,

        "Training Time (s)": training_time
    })

In [ ]:
df_rf  = pd.DataFrame(results_rf)

#final_df = pd.concat([df_cat, df_lr, df_dt, df_rf])

final_df = pd.concat([df_rf])


In [ ]:
print("\n===== Random Forest MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLANATION (Random Forest FINAL)
# ==============================

print("\n--- SHAP EXPLANATION (Random Forest) ---")

import shap
import pandas as pd
import numpy as np

# -------------------------------
# 0. PREPARE DATA (IMPORTANT)
# -------------------------------
# Ensure DataFrame + correct feature names
X_test_df = pd.DataFrame(X_test, columns=X.columns).astype(float)

# 🔥 Use small sample (avoid slow / crash)
X_sample = X_test_df.sample(min(50, len(X_test_df)), random_state=42)

# -------------------------------
# 1. EXPLAINER (TREE MODEL)
# -------------------------------
explainer = shap.TreeExplainer(best_rf)

# -------------------------------
# 2. SHAP VALUES
# -------------------------------
shap_values = explainer.shap_values(X_sample)

# Handle binary classification properly
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_class1 = shap_values[:, :, 1]
else:
    shap_values_class1 = shap_values

# -------------------------------
# 3. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_sample, plot_type="bar")

# -------------------------------
# 4. DETAILED SUMMARY (optional)
# -------------------------------
# shap.summary_plot(shap_values_class1, X_sample)

# -------------------------------
# 5. DEPENDENCE PLOT (FIXED)
# -------------------------------
shap.dependence_plot(
    0,                      # use index instead of column name
    shap_values_class1,
    X_sample
)

# -------------------------------
# 6. LOCAL EXPLANATION (SAFE)
# -------------------------------
base_value = explainer.expected_value

# Fix base value shape
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_sample.iloc[0].values,   # 🔥 FIX
    matplotlib=True
)

In [ ]:
# ==============================
# LIME EXPLANATION (Random Forest - FINAL)
# ==============================

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

print("\n--- LIME EXPLANATION (Random Forest) ---")

# -------------------------------
# ✅ PREPARE DATA (match training pipeline)
# -------------------------------
# If you scaled features for training, use the SAME scaled arrays here:
X_train_df = pd.DataFrame(X_train_f, columns=X.columns).astype(float)
X_test_df  = pd.DataFrame(X_test_f,  columns=X.columns).astype(float)

X_train_np = X_train_df.values
X_test_np  = X_test_df.values

# -------------------------------
# ✅ CREATE EXPLAINER
# -------------------------------
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X.columns.tolist(),
    class_names=["Class 0", "Class 1"],   # adjust labels if needed
    mode="classification",
    discretize_continuous=True
)

# -------------------------------
# ✅ EXPLAIN ONE INSTANCE
# -------------------------------
i = 0  # choose any test index

exp = explainer.explain_instance(
    X_test_np[i],
    best_rf.predict_proba,     # IMPORTANT
    num_features=10
)

# -------------------------------
# ✅ OUTPUT (TEXT)
# -------------------------------
print("\n🔹 Prediction Probabilities:")
print(best_rf.predict_proba(X_test_np[i].reshape(1, -1)))

print("\n🔹 LIME Explanation:")
for feat, w in exp.as_list():
    print(f"{feat}: {w:.4f}")

# -------------------------------
# ✅ VISUAL (if notebook)
# -------------------------------
try:
    exp.show_in_notebook(show_table=True)
except:
    print("Notebook visualization not supported here.")

# -------------------------------
# ✅ SAVE HTML
# -------------------------------
exp.save_to_file("/content/lime_rf_explanation.html")
print("\n📁 Saved: /content/lime_rf_explanation.html")

# **Decision Tree**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [ ]:
# ==============================
# FAST DECISION TREE TUNING
# ==============================

# Smaller parameter space
params_dt = {
    "max_depth": [5, 10],
    "min_samples_split": [2],
    "min_samples_leaf": [1],
    "criterion": ["gini"]
}

# Faster CV
cv_fast = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

# Fast Decision Tree
dt = DecisionTreeClassifier(
    random_state=42,
    class_weight="balanced"
)

# Randomized Search
rscv_dt = RandomizedSearchCV(
    estimator=dt,
    param_distributions=params_dt,
    n_iter=1,          # ultra fast
    scoring='f1',
    cv=cv_fast,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

# Train
rscv_dt.fit(X_train_ad, y_train_ad)

# Best Model
best_dt = rscv_dt.best_estimator_

In [ ]:
print("\n🌳 Decision Tree (Tuned)")

y_pred_dt = best_dt.predict(X_test_scaled)
y_prob_dt = best_dt.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_dt, y_prob_dt)

print("Best Params (DT):", rscv_dt.best_params_)

In [ ]:
results_dt = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    # ==============================
    # SPLIT DATA
    # ==============================
    X_train_f = X.iloc[train_idx]
    X_test_f  = X.iloc[test_idx]

    y_train_f = y.iloc[train_idx]
    y_test_f  = y.iloc[test_idx]

    # ==============================
    # REMOVE SCALING
    # Decision Tree doesn't need scaling
    # ==============================

    # ==============================
    # FAST OVERSAMPLING
    # ==============================
    ros = RandomOverSampler(random_state=42)

    X_train_f, y_train_f = ros.fit_resample(
        X_train_f,
        y_train_f
    )

    # ==============================
    # TRAIN
    # ==============================
    start_time = time.time()

    best_dt.fit(X_train_f, y_train_f)

    training_time = time.time() - start_time

    # ==============================
    # PREDICT
    # ==============================
    y_pred = best_dt.predict(X_test_f)

    y_prob = best_dt.predict_proba(X_test_f)[:, 1]

    # ==============================
    # METRICS
    # ==============================
    acc = accuracy_score(y_test_f, y_pred)

    prec = precision_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_f,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)

    fpr = fp / (fp + tn)

    gm = np.sqrt(rec * specificity)

    auc = roc_auc_score(y_test_f, y_prob)

    mcc = matthews_corrcoef(y_test_f, y_pred)

    kappa = cohen_kappa_score(y_test_f, y_pred)

    balanced_acc = (rec + specificity) / 2

    # ==============================
    # STORE RESULTS
    # ==============================
    results_dt.append({

        "Fold": fold + 1,

        "Classifier": "Decision Tree",

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "Specificity": specificity,

        "F1": f1,

        "GM": gm,

        "FPR": fpr,

        "AUC": auc,

        "MCC": mcc,

        "Kappa": kappa,

        "Balanced Accuracy": balanced_acc,

        "Training Time (s)": training_time
    })


In [ ]:
#All Model
#df_cat = pd.DataFrame(results1)
#df_lr  = pd.DataFrame(results_lr)
df_dt  = pd.DataFrame(results_dt)

final_df = pd.concat([ df_dt])

In [ ]:
#Print Comparison
print("\n===== Decision Tree MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
# ==============================
# SHAP EXPLANATION (Decision Tree FINAL)
# ==============================

print("\n--- SHAP EXPLANATION (Decision Tree) ---")

import shap
import pandas as pd
import numpy as np

# -------------------------------
# 0. PREPARE DATA
# -------------------------------
X_test_df = pd.DataFrame(X_test, columns=X.columns).astype(float)

# 🔥 Use small sample (important for speed)
X_sample = X_test_df.sample(min(50, len(X_test_df)), random_state=42)

# -------------------------------
# 1. EXPLAINER (TREE MODEL)
# -------------------------------
explainer = shap.TreeExplainer(best_dt)   # ← Decision Tree model

# -------------------------------
# 2. SHAP VALUES
# -------------------------------
shap_values = explainer.shap_values(X_sample)

# Handle classification output
if isinstance(shap_values, list):
    shap_values_class1 = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_values_class1 = shap_values[:, :, 1]
else:
    shap_values_class1 = shap_values

# -------------------------------
# 3. GLOBAL FEATURE IMPORTANCE
# -------------------------------
shap.summary_plot(shap_values_class1, X_sample, plot_type="bar")

# -------------------------------
# 4. DEPENDENCE PLOT (SAFE)
# -------------------------------
shap.dependence_plot(
    0,                     # feature index (safe)
    shap_values_class1,
    X_sample
)

# -------------------------------
# 5. LOCAL EXPLANATION
# -------------------------------
base_value = explainer.expected_value

# Fix base value
if isinstance(base_value, (list, np.ndarray)):
    base_value = base_value[1]

shap.force_plot(
    base_value,
    shap_values_class1[0],
    X_sample.iloc[0].values,
    matplotlib=True
)

# **K Nearest Neighbour **

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

In [ ]:
params_knn = {

    "n_neighbors": [5],

    "weights": ["distance"],

    "metric": ["euclidean"]
}

# Faster CV
cv_fast = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

# Fast KNN
knn = KNeighborsClassifier(
    n_jobs=-1
)

# Randomized Search
rscv_knn = RandomizedSearchCV(

    estimator=knn,

    param_distributions=params_knn,

    n_iter=1,          # ultra fast

    scoring='f1',

    cv=cv_fast,

    verbose=1,

    n_jobs=-1,

    random_state=42
)

# Train
rscv_knn.fit(X_train_ad, y_train_ad)

# Best Model
best_knn = rscv_knn.best_estimator_

In [ ]:
print("\n KNN (Tuned)")

y_pred_knn = best_knn.predict(X_test_scaled)
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_knn, y_prob_knn)

print("Best Params (KNN):", rscv_knn.best_params_)

In [ ]:
results_knn = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    # ==============================
    # SPLIT DATA
    # ==============================
    X_train_f = X.iloc[train_idx]

    X_test_f = X.iloc[test_idx]

    y_train_f = y.iloc[train_idx]

    y_test_f = y.iloc[test_idx]

    # ==============================
    # SCALING (IMPORTANT FOR KNN)
    # ==============================
    scaler = StandardScaler()

    X_train_f = scaler.fit_transform(X_train_f)

    X_test_f = scaler.transform(X_test_f)

    # ==============================
    # FAST OVERSAMPLING
    # ==============================
    ros = RandomOverSampler(random_state=42)

    X_train_f, y_train_f = ros.fit_resample(
        X_train_f,
        y_train_f
    )

    # ==============================
    # TRAIN
    # ==============================
    start_time = time.time()

    best_knn.fit(X_train_f, y_train_f)

    training_time = time.time() - start_time

    # ==============================
    # PREDICT
    # ==============================
    y_pred = best_knn.predict(X_test_f)

    # ==============================
    # FAST METRICS
    # ==============================
    acc = accuracy_score(y_test_f, y_pred)

    prec = precision_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_f,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)

    balanced_acc = (rec + specificity) / 2

    # ==============================
    # STORE RESULTS
    # ==============================
    results_knn.append({

        "Fold": fold + 1,

        "Classifier": "KNN",

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "Specificity": specificity,

        "F1": f1,

        "Balanced Accuracy": balanced_acc,

        "Training Time (s)": training_time
    })

In [ ]:
df_knn = pd.DataFrame(results_knn)

final_df = pd.concat([df_knn])

In [ ]:
print("\n===== KNN MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
print("\n--- SHAP EXPLANATION (KNN FINAL - FIXED) ---")

import shap
import numpy as np
import pandas as pd

# -------------------------------
# STEP 1: Prepare Data
# -------------------------------
feature_names = X.columns.tolist()

X_train_df = pd.DataFrame(X_train, columns=feature_names).astype(float)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names).astype(float)

# -------------------------------
# STEP 2: Speed Control
# -------------------------------
X_background = X_train_df.sample(5, random_state=42)
X_sample     = X_test_df.sample(2, random_state=42)

# -------------------------------
# STEP 3: SHAP Explainer
# -------------------------------
explainer = shap.KernelExplainer(
    best_knn.predict_proba,
    X_background
)

# -------------------------------
# STEP 4: SHAP Values
# -------------------------------
shap_values = explainer.shap_values(X_sample)

# -------------------------------
# STEP 5: FIX SHAPE (IMPORTANT)
# -------------------------------
if isinstance(shap_values, list):
    shap_vals = shap_values[1]               # class 1
    base_val  = explainer.expected_value[1]
else:
    if len(shap_values.shape) == 3:
        shap_vals = shap_values[:, :, 1]
        base_val  = explainer.expected_value[1]
    else:
        shap_vals = shap_values
        base_val  = explainer.expected_value

# Ensure scalar base value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(np.array(base_val).flatten()[0])

# Debug check (optional)
print("SHAP shape:", shap_vals.shape)
print("X_sample shape:", X_sample.shape)

# -------------------------------
# STEP 6: GLOBAL IMPORTANCE
# -------------------------------
shap.summary_plot(shap_vals, X_sample, plot_type="bar")

# -------------------------------
# STEP 7: FORCE PLOT (SAFE)
# -------------------------------
sample = X_sample.iloc[0]
shap_val = shap_vals[0]

# Ensure correct shape
shap_val = np.array(shap_val).reshape(-1)
sample_vals = sample.values.reshape(-1)

print("Single SHAP:", shap_val.shape)
print("Single sample:", sample_vals.shape)

shap.force_plot(
    base_val,
    shap_val,
    sample_vals,
    matplotlib=True
)

In [ ]:
# ==============================
# LIME EXPLANATION (KNN FINAL)
# ==============================

from lime.lime_tabular import LimeTabularExplainer
import numpy as np
import pandas as pd

print("\n--- LIME EXPLANATION (KNN) ---")

# -------------------------------
# 0. PREPARE DATA (match training pipeline)
# -------------------------------
# If you scaled before training KNN, use the SAME scaled arrays here
X_train_df = pd.DataFrame(X_train, columns=X.columns).astype(float)
X_test_df  = pd.DataFrame(X_test,  columns=X.columns).astype(float)

# Ensure no NaN
X_train_df = X_train_df.fillna(0)
X_test_df  = X_test_df.fillna(0)

X_train_np = X_train_df.values
X_test_np  = X_test_df.values

# -------------------------------
# 1. CREATE EXPLAINER
# -------------------------------
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=X.columns.tolist(),
    class_names=["Class 0", "Class 1"],  # adjust if needed
    mode="classification",
    discretize_continuous=True
)

# -------------------------------
# 2. EXPLAIN ONE INSTANCE
# -------------------------------
i = 0  # choose any test index

exp = explainer.explain_instance(
    X_test_np[i],
    best_knn.predict_proba,     # IMPORTANT
    num_features=10
)

# -------------------------------
# 3. OUTPUT (TEXT)
# -------------------------------
print("\n🔹 Prediction Probabilities:")
print(best_knn.predict_proba(X_test_np[i].reshape(1, -1)))

print("\n🔹 LIME Explanation:")
for feat, w in exp.as_list():
    print(f"{feat}: {w:.4f}")

# -------------------------------
# 4. VISUAL (NOTEBOOK)
# -------------------------------
try:
    exp.show_in_notebook(show_table=True)
except:
    print("Notebook visualization not supported here.")

# -------------------------------
# 5. SAVE HTML
# -------------------------------
exp.save_to_file("/content/lime_knn_explanation.html")
print("\n📁 Saved: /content/lime_knn_explanation.html")

# ** Multi Layer Perceptron**

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold

In [ ]:
params_mlp = {

    "hidden_layer_sizes": [(50,)],

    "alpha": [0.001],

    "activation": ["relu"]
}

# Fast MLP
mlp = MLPClassifier(

    hidden_layer_sizes=(50,),

    activation='relu',

    solver='adam',

    alpha=0.001,

    batch_size=256,          # faster training

    learning_rate='adaptive',

    max_iter=80,             # lower

    early_stopping=True,     # huge speed boost

    validation_fraction=0.1,

    n_iter_no_change=3,      # faster stop

    random_state=42
)

# Faster CV
cv_fast = StratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

# Random Search
rscv_mlp = RandomizedSearchCV(

    estimator=mlp,

    param_distributions=params_mlp,

    n_iter=1,                # ultra fast

    scoring='f1',

    cv=cv_fast,

    n_jobs=-1,

    verbose=1,

    random_state=42
)

# Train
rscv_mlp.fit(X_train_ad, y_train_ad)

# Best Model
best_mlp = rscv_mlp.best_estimator_

In [ ]:
print("\n MLP (Tuned)")

y_pred_mlp = best_mlp.predict(X_test_scaled)
y_prob_mlp = best_mlp.predict_proba(X_test_scaled)[:,1]

compute_metrics(y_test, y_pred_mlp, y_prob_mlp)

print("Best Params (MLP):", rscv_mlp.best_params_)

In [ ]:
results_mlp = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):

    # ==============================
    # SPLIT DATA
    # ==============================
    X_train_f = X.iloc[train_idx]

    X_test_f = X.iloc[test_idx]

    y_train_f = y.iloc[train_idx]

    y_test_f = y.iloc[test_idx]

    # ==============================
    # SCALING (IMPORTANT FOR MLP)
    # ==============================
    scaler = StandardScaler()

    X_train_f = scaler.fit_transform(X_train_f)

    X_test_f = scaler.transform(X_test_f)

    # ==============================
    # FAST OVERSAMPLING
    # ==============================
    ros = RandomOverSampler(random_state=42)

    X_train_f, y_train_f = ros.fit_resample(
        X_train_f,
        y_train_f
    )

    # ==============================
    # TRAIN
    # ==============================
    start_time = time.time()

    best_mlp.fit(X_train_f, y_train_f)

    training_time = time.time() - start_time

    # ==============================
    # PREDICT
    # ==============================
    y_pred = best_mlp.predict(X_test_f)

    # ==============================
    # FAST METRICS
    # ==============================
    acc = accuracy_score(y_test_f, y_pred)

    prec = precision_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    rec = recall_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test_f,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test_f,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)

    balanced_acc = (rec + specificity) / 2

    # ==============================
    # STORE RESULTS
    # ==============================
    results_mlp.append({

        "Fold": fold + 1,

        "Classifier": "MLP",

        "Accuracy": acc,

        "Precision": prec,

        "Recall": rec,

        "Specificity": specificity,

        "F1": f1,

        "Balanced Accuracy": balanced_acc,

        "Training Time (s)": training_time
    })

In [ ]:
df_mlp = pd.DataFrame(results_mlp)

final_df = pd.concat([ df_mlp])

In [ ]:
print("\n===== MLP MODELS =====")
print(final_df.to_string(index=False))

print("\n===== AVERAGE COMPARISON =====")
print(final_df.groupby("Classifier").mean(numeric_only=True))

In [ ]:
print("\n--- SHAP EXPLANATION (MLP FINAL - ERROR FREE) ---")

import shap
import numpy as np
import pandas as pd

# -------------------------------
# STEP 1: Prepare Data
# -------------------------------
feature_names = X.columns.tolist()

X_train_df = pd.DataFrame(X_train, columns=feature_names).astype(float)
X_test_df  = pd.DataFrame(X_test,  columns=feature_names).astype(float)

# -------------------------------
# STEP 2: Speed Control
# -------------------------------
X_background = X_train_df.sample(10, random_state=42)
X_sample     = X_test_df.sample(5, random_state=42)

# -------------------------------
# STEP 3: SHAP Explainer
# -------------------------------
explainer = shap.KernelExplainer(
    best_mlp.predict_proba,
    X_background
)

# -------------------------------
# STEP 4: SHAP Values
# -------------------------------
shap_values = explainer.shap_values(X_sample)

# -------------------------------
# STEP 5: FIX SHAPE (CRITICAL)
# -------------------------------
if isinstance(shap_values, list):
    shap_vals = shap_values[1]   # binary → class 1
    base_val  = explainer.expected_value[1]
else:
    if len(shap_values.shape) == 3:
        shap_vals = shap_values[:, :, 1]   # pick class 1
        base_val  = explainer.expected_value[1]
    else:
        shap_vals = shap_values
        base_val  = explainer.expected_value

# Ensure scalar base value
if isinstance(base_val, (list, np.ndarray)):
    base_val = float(np.array(base_val).flatten()[0])

# Debug (optional)
print("SHAP shape:", shap_vals.shape)
print("X_sample shape:", X_sample.shape)

# -------------------------------
# STEP 6: GLOBAL IMPORTANCE
# -------------------------------
shap.summary_plot(shap_vals, X_sample, plot_type="bar")

# -------------------------------
# STEP 7: LOCAL EXPLANATION
# -------------------------------
exp = shap.Explanation(
    values=shap_vals[0],
    base_values=base_val,
    data=X_sample.iloc[0],
    feature_names=feature_names
)

# Stable plot (no errors)
shap.plots.waterfall(exp)

# Optional interactive plot
# shap.plots.force(exp)

# -------------------------------
# FORCE PLOT (FINAL - CORRECT)
# -------------------------------

# Select single sample
sample = X_sample.iloc[0]

# Get correct SHAP values (class 1)
if isinstance(shap_values, list):
    shap_val = shap_values[1][0]   # class 1
    base_val = explainer.expected_value[1]
else:
    if len(shap_values.shape) == 3:
        shap_val = shap_values[0, :, 1]
        base_val = explainer.expected_value[1]
    else:
        shap_val = shap_values[0]
        base_val = explainer.expected_value

# Ensure correct shape
shap_val = np.array(shap_val).reshape(-1)
sample_vals = sample.values.reshape(-1)

# Final check
print("SHAP:", shap_val.shape)
print("Features:", sample_vals.shape)

# Plot
shap.force_plot(
    float(base_val),
    shap_val,
    sample_vals,
    matplotlib=True
)

In [ ]:
print("\n--- LIME EXPLANATION (MLP) ---")

from lime.lime_tabular import LimeTabularExplainer
import numpy as np

# -------------------------------
# STEP 1: Prepare data
# -------------------------------
feature_names = X.columns.tolist()
class_names = ['Class 0', 'Class 1']  # change if needed

# Convert to numpy (LIME needs numpy)
X_train_np = np.array(X_train)
X_test_np  = np.array(X_test)

# -------------------------------
# STEP 2: Create Explainer
# -------------------------------
explainer = LimeTabularExplainer(
    training_data=X_train_np,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification'
)

# -------------------------------
# STEP 3: Select sample
# -------------------------------
i = 0   # you can change index
sample = X_test_np[i]

# -------------------------------
# STEP 4: Explain prediction
# -------------------------------
exp = explainer.explain_instance(
    data_row=sample,
    predict_fn=best_mlp.predict_proba,
    num_features=10
)

# -------------------------------
# STEP 5: Show explanation
# -------------------------------
exp.show_in_notebook(show_table=True)

# -------------------------------
# STEP 6: Print explanation
# -------------------------------
print("\nFeature Contributions:")
for feature, weight in exp.as_list():
    print(f"{feature}: {weight:.4f}")

# **CREATE FINAL RESULTS FILE**

In [ ]:
# ==============================
# CREATE FINAL RESULTS FILE
# ==============================

import pandas as pd

# ==============================
# CONVERT LIST TO DATAFRAME
# ==============================
results1 = pd.DataFrame(results1)

# ==============================
# COMBINE ALL RESULT DATAFRAMES
# ==============================
df_all = pd.concat([

    pd.DataFrame(results_lr),

    pd.DataFrame(results_dt),

    pd.DataFrame(results_rf),

    pd.DataFrame(results_gbm),

    pd.DataFrame(results_xgb),

    pd.DataFrame(results_lgb),

    pd.DataFrame(results1),      # CatBoost

    pd.DataFrame(results_knn),

    pd.DataFrame(results_mlp)

], ignore_index=True)

# ==============================
# SAVE CSV FILE
# ==============================
file_path = "/content/final_results.csv"

df_all.to_csv(file_path, index=False)

# ==============================
# OUTPUT
# ==============================
print("✅ Results file created successfully!")

print("\n📁 File saved at:")
print(file_path)

print("\n==============================")
print("FIRST 5 ROWS")
print("==============================")

print(df_all.head())

print("\n==============================")
print("COLUMNS")
print("==============================")

print(df_all.columns)

print("\n==============================")
print("DATA SHAPE")
print("==============================")

print(df_all.shape)

# ** wilcoxon**

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon
from itertools import combinations
from statsmodels.stats.multitest import multipletests

# ==============================
# LOAD FILE
# ==============================
file_path = "/content/final_results.csv"
df = pd.read_csv(file_path)

# -------------------------------
# CHECK REQUIRED COLUMNS
# -------------------------------
required_cols = ["Fold", "Classifier"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"❌ Missing column: {col}")

# -------------------------------
# METRICS
# -------------------------------
metrics = [
    "Accuracy", "AUC", "F1", "FPR", "GM",
    "Kappa", "MCC", "Precision", "Recall", "Specificity"
]

lower_is_better = ["FPR"]

all_results = []

# ==============================
# LOOP METRICS
# ==============================
for metric in metrics:

    if metric not in df.columns:
        print(f"⚠️ Skipping: {metric}")
        continue

    print(f"\nProcessing: {metric}")

    pivot = df.pivot(index="Fold", columns="Classifier", values=metric)

    models = pivot.columns

    # -------------------------------
    # ✅ UNIQUE PAIRWISE (NO DUPLICATE)
    # -------------------------------
    for m1, m2 in combinations(models, 2):

        scores1 = pivot[m1].dropna()
        scores2 = pivot[m2].dropna()

        # Align folds
        common_idx = scores1.index.intersection(scores2.index)
        scores1 = scores1.loc[common_idx]
        scores2 = scores2.loc[common_idx]

        if len(scores1) < 5:
            continue

        # Wilcoxon test
        stat, p = wilcoxon(scores1, scores2)

        mean1 = scores1.mean()
        mean2 = scores2.mean()

        # Determine better model
        if metric in lower_is_better:
            better = m1 if mean1 < mean2 else m2
        else:
            better = m1 if mean1 > mean2 else m2

        all_results.append({
            "Metric": metric,
            "Model 1": m1,
            "Model 2": m2,
            "Mean 1": round(mean1, 4),
            "Mean 2": round(mean2, 4),
            "Better Model": better,
            "Statistic": round(stat, 4),
            "p-value": p
        })

# ==============================
# CREATE DATAFRAME
# ==============================
results_df = pd.DataFrame(all_results)

# -------------------------------
# HOLM CORRECTION
# -------------------------------
reject, p_adj, _, _ = multipletests(results_df["p-value"], method='holm')

results_df["Adjusted p-value"] = p_adj
results_df["Significant"] = ["Yes" if r else "No" for r in reject]

# -------------------------------
# ROUND VALUES
# -------------------------------
results_df["p-value"] = results_df["p-value"].round(5)
results_df["Adjusted p-value"] = results_df["Adjusted p-value"].round(5)

# -------------------------------
# SORT RESULTS
# -------------------------------
results_df = results_df.sort_values(by=["Metric", "Adjusted p-value"])

# ==============================
# SAVE OUTPUT
# ==============================
output_file = "/content/wilcoxon_final_output.xlsx"
results_df.to_excel(output_file, index=False)

print("\n✅ FINAL RESULT SAVED:", output_file)

# 🔥 PRINT OUTPUT
print("\n===== FINAL RESULTS =====")
print(results_df.to_string(index=False))

# **Topcis**

In [ ]:
import pandas as pd

# ==============================
# LOAD DATA
# ==============================
file_path = "/content/final_results.csv"
df = pd.read_csv(file_path)

# -------------------------------
# METRICS
# -------------------------------
metrics = [
    "Accuracy", "Precision", "Recall", "Specificity",
    "F1", "GM", "FPR", "AUC", "MCC", "Kappa", "Balanced Accuracy"
]

lower_is_better = ["FPR"]

# -------------------------------
# MEAN PERFORMANCE PER MODEL
# -------------------------------
df_mean = df.groupby("Classifier")[metrics].mean().reset_index()

# -------------------------------
# LONG FORMAT
# -------------------------------
df_long = df_mean.melt(
    id_vars="Classifier",
    var_name="Metric",
    value_name="Score"
)

# Rename column for your format
df_long.rename(columns={"Classifier": "Model"}, inplace=True)

# -------------------------------
# RANKING (FIXED)
# -------------------------------
df_long["Rank"] = df_long.groupby("Metric")["Score"].rank(
    method="average",
    ascending=False
)

# FIX: FPR (lower is better)
mask = df_long["Metric"] == "FPR"
df_long.loc[mask, "Rank"] = (
    df_long[mask]
    .groupby("Metric")["Score"]
    .rank(method="average", ascending=True)
)

# -------------------------------
# SORT OUTPUT
# -------------------------------
df_long = df_long.sort_values(by=["Metric", "Rank"])

# -------------------------------
# SAVE FILE
# -------------------------------
output_file = "/content/final_metric_ranking.xlsx"
df_long.to_excel(output_file, index=False)

# -------------------------------
# PRINT OUTPUT
# -------------------------------
print("\n✅ FINAL RESULT:\n")
print(df_long.to_string(index=False))

print("\n📁 Saved file:", output_file)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.preprocessing import LabelEncoder

from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.tree import DecisionTreeClassifier

from sklearn.neighbors import KNeighborsClassifier

from sklearn.linear_model import LogisticRegression

from scipy.stats import friedmanchisquare

# =========================================================
# LOAD DATASET
# =========================================================

df = pd.read_csv(
    "/content/INCART 2-lead Arrhythmia Database.csv"
)

print("Dataset Shape :", df.shape)

# =========================================================
# TARGET COLUMN
# =========================================================

target_col = "type"

print("\nTarget Column :", target_col)

# =========================================================
# FEATURES & TARGET
# =========================================================

X = df.drop(target_col, axis=1)

y = df[target_col]

# =========================================================
# ENCODE TARGET LABEL
# =========================================================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

# =========================================================
# CONVERT CATEGORICAL FEATURES
# =========================================================

for col in X.columns:

    if X[col].dtype == 'object':

        le = LabelEncoder()

        X[col] = le.fit_transform(
            X[col].astype(str)
        )

# =========================================================
# HANDLE MISSING VALUES
# =========================================================

imputer = SimpleImputer(strategy='mean')

X = imputer.fit_transform(X)

# =========================================================
# DEFINE MODELS
# =========================================================

models = {

    "Random Forest": RandomForestClassifier(),

    "Decision Tree": DecisionTreeClassifier(),

    "KNN": KNeighborsClassifier(),

    "Logistic Regression": LogisticRegression(
        max_iter=1000
    )
}

# =========================================================
# CROSS VALIDATION
# =========================================================

kfold = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42
)

results = {}

print("\n========== MODEL ACCURACY ==========\n")

for name, model in models.items():

    scores = cross_val_score(

        model,

        X,

        y,

        cv=kfold,

        scoring='accuracy',

        n_jobs=-1
    )

    results[name] = scores

    print(name)

    print("Accuracy Scores :", scores)

    print("Average Accuracy :", np.mean(scores))

    print("-----------------------------------")

# =========================================================
# FRIEDMAN TEST
# =========================================================

statistic, p_value = friedmanchisquare(

    results["Random Forest"],

    results["Decision Tree"],

    results["KNN"],

    results["Logistic Regression"]
)

# =========================================================
# FINAL RESULT
# =========================================================

print("\n===================================")

print("         FRIEDMAN TEST")

print("===================================\n")

print("Friedman Statistic :", statistic)

print("P-value            :", p_value)

alpha = 0.05

print("\n========== INTERPRETATION ==========\n")

if p_value < alpha:

    print("Reject Null Hypothesis (H0)")

    print("There is a significant difference")

    print("between the machine learning models.")

else:

    print("Fail to Reject Null Hypothesis (H0)")

    print("No significant difference")

    print("between the machine learning models.")

In [ ]:
# =========================================================
# STORE MODEL RESULTS
# =========================================================

results_table = []

for name, scores in results.items():

    results_table.append({

        "Model": name,

        "Fold 1": scores[0],

        "Fold 2": scores[1],

        "Fold 3": scores[2],

        "Fold 4": scores[3],

        "Fold 5": scores[4],

        "Average Accuracy": np.mean(scores)
    })

# Convert to DataFrame
results_df = pd.DataFrame(results_table)

# =========================================================
# STORE FRIEDMAN TEST RESULT
# =========================================================

friedman_df = pd.DataFrame({

    "Friedman Statistic": [statistic],

    "P-value": [p_value],

    "Decision": [

        "Reject H0"
        if p_value < 0.05
        else "Fail to Reject H0"
    ]
})

# =========================================================
# SAVE CSV FILES
# =========================================================

results_df.to_csv(
    "/content/model_accuracy_results.csv",
    index=False
)

friedman_df.to_csv(
    "/content/friedman_test_result.csv",
    index=False
)

# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\n========== MODEL RESULTS ==========\n")

print(results_df)

print("\n========== FRIEDMAN TEST ==========\n")

print(friedman_df)

print("\n✅ Files Saved Successfully!")

print("\nSaved Files:")

print("/content/model_accuracy_results.csv")

print("/content/friedman_test_result.csv")

In [ ]:
import json

# ==========================================
# NOTEBOOK PATH
# ==========================================
notebook_path = "/content/Copy_of_NewDataSet__Allmodel_comparison_lime (2).ipynb"

# ==========================================
# LOAD NOTEBOOK
# ==========================================
with open(notebook_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# ==========================================
# REMOVE ALL WIDGET REFERENCES RECURSIVELY
# ==========================================

def remove_widgets(obj):

    if isinstance(obj, dict):

        # Remove widgets key completely
        obj.pop("widgets", None)

        # Remove widget mime types
        obj.pop("application/vnd.jupyter.widget-view+json", None)

        for key in list(obj.keys()):
            remove_widgets(obj[key])

    elif isinstance(obj, list):

        for item in obj:
            remove_widgets(item)

# Run cleanup
remove_widgets(nb)

# ==========================================
# OPTIONAL: CLEAR OUTPUTS (BEST)
# ==========================================

for cell in nb.get("cells", []):

    if cell.get("cell_type") == "code":
        cell["outputs"] = []
        cell["execution_count"] = None

# ==========================================
# SAVE CLEAN NOTEBOOK
# ==========================================

fixed_path = "/content/fixed_notebook.ipynb"

with open(fixed_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, indent=1)

print("✅ Fully cleaned notebook saved!")
print("📁 File:", fixed_path)